# Singular Value Decomposition (SVD) Implementation

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand SVD fundamentals and mathematical basis
- Implement SVD in Python using NumPy
- Apply SVD for matrix decomposition, data compression, and noise reduction
- Analyze properties and importance of SVD
- Compare SVD with PCA in data analysis

## 🔗 Prerequisites

- ✅ Unit 1: Linear Algebra (matrices, eigenvalues)
- ✅ Unit 4: PCA implementation
- ✅ Understanding of matrix operations

---

This notebook covers practical activities from **Course 03, Unit 4**:
- Applying SVD for matrix decomposition, data compression, and noise reduction
- Comparing SVD and PCA in data analysis

---

## Introduction

**Singular Value Decomposition (SVD)** is a fundamental matrix factorization technique that decomposes any matrix into three matrices, revealing the underlying structure of the data.

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Setup: NumPy/Matplotlib plus sklearn's TruncatedSVD and a blob-data generator for the demos.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.datasets import make_blobs

# Confirm the environment is ready before any math starts.
print("✅ Libraries imported!")


✅ Libraries imported!


## Part 1: Understanding SVD Fundamentals

SVD decomposes a matrix A into: A = U × Σ × V^T


In [2]:
# Generate sample data matrix
A = np.random.rand(10, 8)
print("=" * 60)
print("SVD Fundamentals")
print("=" * 60)
print(f"Original matrix A shape: {A.shape}")

# Compute SVD
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# Create diagonal matrix from singular values
Sigma = np.diag(s)

print(f"\nU shape: {U.shape} (left singular vectors)")
print(f"Σ (Sigma) shape: {Sigma.shape} (singular values)")
print(f"V^T shape: {Vt.shape} (right singular vectors)")

# Verify decomposition: A = U × Σ × V^T
A_reconstructed = U @ Sigma @ Vt
reconstruction_error = np.linalg.norm(A - A_reconstructed)

print(f"\nReconstruction error: {reconstruction_error:.2e}")
print(f"✅ SVD decomposition successful! (error ≈ 0)")

print(f"\nSingular values: {s[:5]} ... (showing first 5)")


SVD Fundamentals
Original matrix A shape: (10, 8)

U shape: (10, 8) (left singular vectors)
Σ (Sigma) shape: (8, 8) (singular values)
V^T shape: (8, 8) (right singular vectors)

Reconstruction error: 4.48e-15
✅ SVD decomposition successful! (error ≈ 0)

Singular values: [3.91299488 1.2431032  1.06722776 1.03829086 0.8474028 ] ... (showing first 5)


## Part 2: SVD for Data Compression

Use SVD to compress data while preserving most information.


In [3]:
# Rebuild the matrix from only its top-k singular triplets and measure what compression costs.
# Because singular values are sorted by importance, a few of them carry most of the information.

# SVD for compression: Keep only top k singular values
k = 3  # Keep top 3 components

# Slice all three factors down to rank k.
U_k = U[:, :k]
Sigma_k = Sigma[:k, :k]
Vt_k = Vt[:k, :]

A_compressed = U_k @ Sigma_k @ Vt_k

# Storage: three small factors instead of the full matrix; 'information' = share of squared singular values.
compression_ratio = (U_k.size + Sigma_k.size + Vt_k.size) / A.size
info_retained = (s[:k]**2).sum() / (s**2).sum()

print("=" * 60)
print("SVD for Data Compression")
print("=" * 60)
print(f"Original matrix size: {A.size} elements")
print(f"Compressed representation: {U_k.size + Sigma_k.size + Vt_k.size} elements")
print(f"Compression ratio: {compression_ratio:.2%}")
print(f"Information retained: {info_retained:.2%}")

# How far is the rank-k reconstruction from the original?
reconstruction_error_compressed = np.linalg.norm(A - A_compressed)
print(f"Reconstruction error (k={k}): {reconstruction_error_compressed:.4f}")

print("\n✅ SVD allows compression while preserving most information!")


SVD for Data Compression
Original matrix size: 80 elements
Compressed representation: 63 elements
Compression ratio: 78.75%
Information retained: 88.27%
Reconstruction error (k=3): 1.5463

✅ SVD allows compression while preserving most information!


## Part 3: SVD vs PCA Comparison

Compare SVD and PCA for dimensionality reduction.


In [4]:
# Cross-check SVD against PCA on the same dataset — for centered data they are near-twins.
# TruncatedSVD earns its keep on sparse matrices, where PCA's centering step would destroy sparsity.

try:
    from sklearn.decomposition import PCA
    
    # Generate sample dataset
    X, y = make_blobs(n_samples=100, n_features=10, centers=3, random_state=42)
    
    # Apply PCA
    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(X)
    
    # Apply SVD (TruncatedSVD is similar to PCA for centered data)
    svd = TruncatedSVD(n_components=3)
    X_svd = svd.fit_transform(X)
    
    print("=" * 60)
    print("SVD vs PCA Comparison")
    print("=" * 60)
    print(f"Original data shape: {X.shape}")
    print(f"PCA reduced shape: {X_pca.shape}")
    print(f"SVD reduced shape: {X_svd.shape}")
    
    print(f"\nPCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")
    print(f"SVD explained variance: {svd.explained_variance_ratio_.sum():.2%}")
    
    print("\n✅ Key Differences:")
    print("  - PCA requires centered data (mean = 0)")
    print("  - SVD works on raw data matrices")
    print("  - For centered data, PCA ≈ SVD")
    print("  - SVD is more general and applicable to sparse matrices")
    
except ImportError:
    print("Note: Install scikit-learn to compare SVD with PCA")


SVD vs PCA Comparison
Original data shape: (100, 10)
PCA reduced shape: (100, 3)
SVD reduced shape: (100, 3)

PCA explained variance: 97.14%
SVD explained variance: 96.97%

✅ Key Differences:
  - PCA requires centered data (mean = 0)
  - SVD works on raw data matrices
  - For centered data, PCA ≈ SVD
  - SVD is more general and applicable to sparse matrices


## Part 4: SVD for Noise Reduction

Real data is often "low-rank signal + noise". The small singular values mostly capture the noise, so truncating them can **denoise** a matrix: reconstruct with only the top k singular values and compare against the clean signal.

In [5]:
# Denoising demo: bury a rank-3 signal in noise, keep the top 3 singular values, and recover it.
# Truncated SVD works as a filter because signal concentrates in large singular values while noise spreads thin.

np.random.seed(42)

print("=" * 60)
print("SVD for Noise Reduction")
print("=" * 60)

# Build a clean low-rank signal (rank 3) and add noise
rank = 3
A_clean = np.random.randn(20, rank) @ np.random.randn(rank, 15)
A_noisy = A_clean + 0.5 * np.random.randn(20, 15)

# Truncated SVD of the NOISY matrix
U_n, s_n, Vt_n = np.linalg.svd(A_noisy, full_matrices=False)
k_denoise = rank
A_denoised = U_n[:, :k_denoise] @ np.diag(s_n[:k_denoise]) @ Vt_n[:k_denoise, :]

err_noisy = np.linalg.norm(A_noisy - A_clean)
err_denoised = np.linalg.norm(A_denoised - A_clean)

print(f"\nSingular values of the noisy matrix (first 6): {np.round(s_n[:6], 2)}")
print(f"  → the top {rank} are much larger: they carry the signal; the tail is mostly noise")
print(f"\nError vs clean signal BEFORE denoising: {err_noisy:.4f}")
print(f"Error vs clean signal AFTER keeping top {k_denoise} components: {err_denoised:.4f}")
print(f"Error reduced by {(1 - err_denoised / err_noisy) * 100:.1f}%")

if err_denoised < err_noisy:
    print("\n✅ Truncating small singular values removed noise — the reconstruction is CLOSER to the clean signal!")
else:
    print("\n⚠️ Denoising did not help here — try a different k")

SVD for Noise Reduction

Singular values of the noisy matrix (first 6): [17.33 13.4   9.97  3.69  2.84  2.56]
  → the top 3 are much larger: they carry the signal; the tail is mostly noise

Error vs clean signal BEFORE denoising: 8.4493
Error vs clean signal AFTER keeping top 3 components: 4.9119
Error reduced by 41.9%

✅ Truncating small singular values removed noise — the reconstruction is CLOSER to the clean signal!


## Summary

### Key Concepts:
1. **SVD Decomposition**: A = U × Σ × V^T
   - U: Left singular vectors
   - Σ: Singular values (diagonal matrix)
   - V^T: Right singular vectors

2. **Data Compression**: Keep top k singular values to compress data
3. **Noise Reduction**: Remove small singular values to reduce noise
4. **SVD vs PCA**: 
   - SVD works on raw matrices
   - PCA requires centered data
   - For centered data, PCA ≈ SVD

### Applications:
- **Image Compression**: Reduce storage while preserving quality
- **Recommendation Systems**: Matrix factorization for collaborative filtering
- **Noise Reduction**: Remove noise by truncating small singular values
- **Dimensionality Reduction**: Reduce dimensions while preserving information

**Reference:** Course 03, Unit 4: "Dimensionality Reduction and Data Representation Techniques" - SVD practical content


## 📚 References

1. Eckart, C., & Young, G. (1936). *The Approximation of One Matrix by Another of Lower Rank*. Psychometrika, 1(3), 211–218.
2. Golub, G., & Kahan, W. (1965). *Calculating the Singular Values and Pseudo-Inverse of a Matrix*. SIAM Journal on Numerical Analysis, Series B, 2(2), 205–224.
3. Halko, N., Martinsson, P.-G., & Tropp, J. A. (2011). *Finding Structure with Randomness: Probabilistic Algorithms for Constructing Approximate Matrix Decompositions*. SIAM Review, 53(2), 217–288. <https://arxiv.org/abs/0909.4061>